# reading plates from the dock camera (simulated)

this is step 3 of the plan in `dock_shipments.ipynb`: a camera sees a truck, reads its plate, and checks it against the schedule before the door opens.

```
plate_video/dock_cam_plates.mp4  (fake dock camera)
        |
   YOLO finds the vehicle  ->  crop the bottom of the vehicle  ->  OCR reads the plate
        |
   same plate read 3 times in a row = one arrival
        |
   find_truck() in dock.db  ->  open (pulse remote_open) or deny, logged with source = yolo
        |
   score it against plate_video/ground_truth.csv
```

YOLO only finds vehicles. it does not read text. the reading is done by EasyOCR.

by default this runs on a **copy** of `dock.db` and does **not** touch the PLC, so you can run it as many times as you want and get the same score. step 2 has a switch (`LIVE`) for the real thing.

## 1. install (once per container)

YOLO (`ultralytics`) and EasyOCR are not in `uv.lock`, so this installs them into the running notebooks container. it takes a few minutes the first time.

it uses `uv` instead of `%pip` because the container's python is set up by uv, which doesn't include pip.

they stay installed until you run `docker compose down`. after that, run this cell again.

`ultralytics` pulls in the desktop version of opencv, which needs screen libraries the container doesn't have. the last two lines swap it for the headless version.

**restart the kernel after this cell** (Kernel → Restart Kernel), then go on to step 2.

In [ ]:
import sys
!uv pip install --python {sys.executable} ultralytics easyocr
!uv pip uninstall --python {sys.executable} opencv-python
!uv pip install --python {sys.executable} --reinstall --no-deps opencv-python-headless

## 2. settings

| setting | meaning |
|---|---|
| `VIDEO` | the video to watch |
| `GROUND_TRUTH` | the answer key that goes with it |
| `LIVE` | `False`: work on a copy of dock.db, don't touch the PLC. `True`: write to dock.db and pulse `remote_open` |
| `SAMPLE_EVERY` | look at every Nth frame. the video is 30 fps, so 10 means 3 looks per second |
| `CONFIRM` | how many reads in a row must agree before it counts as an arrival |
| `OCR_SCALE` | shrink the plate area before OCR. smaller is faster |
| `VEHICLES` | the YOLO classes we care about (these are the standard COCO numbers) |
| `PLATE_PATTERN` | what a plate looks like: 3 letters, a dash, 3 or 4 letters/numbers |
| `PLATE_ZONE` | the part of the picture to read when YOLO doesn't find a vehicle (see step 4) |

In [ ]:
import os
import re
import time
import sqlite3
from collections import Counter
from datetime import datetime

import cv2
import pandas as pd

VIDEO = "plate_video/dock_cam_plates.mp4"
GROUND_TRUTH = "plate_video/ground_truth.csv"
DB_PATH = "dock.db"

LIVE = False

SAMPLE_EVERY = 10
CONFIRM = 2
OCR_SCALE = 0.5   # shrink the crop before OCR. half size reads just as well and is about 4x faster

VEHICLES = {2: "car", 5: "bus", 7: "truck"}
PLATE_PATTERN = re.compile(r"^[A-Z]{3}-[A-Z0-9]{3,4}$")

# where plates show up when a vehicle is stopped at the door, as fractions of the picture
# (left, top, right, bottom). used when YOLO doesn't find the vehicle
PLATE_ZONE = (0.25, 0.40, 0.75, 1.00)

print("live mode" if LIVE else "test mode: copy of dock.db, no PLC")

## 3. load the models

`yolo11n.pt` is the smallest YOLO model, trained on everyday photos (COCO). it downloads the first time (about 5 MB).
EasyOCR also downloads its models the first time (about 100 MB).

`gpu=False` because the container only has the CPU.

In [ ]:
from ultralytics import YOLO
import easyocr

yolo = YOLO("yolo11n.pt")
ocr = easyocr.Reader(["en"], gpu=False, verbose=False)
print("models ready")

## 4. the plate reader

`read_plate(frame)` does one frame:

1. YOLO finds vehicles. we keep the biggest one, since that's the one at the door
2. cut out the bottom 45% of that box, where the front plate is
3. if YOLO found nothing, use the **plate zone** instead: the fixed part of the picture where plates show up when a vehicle is at the door. the camera never moves, so this always works. it also keeps the camera's own clock and label (top of the picture) out of the OCR
4. EasyOCR reads the text. it's only allowed capital letters, numbers and the dash
5. clean it up and keep it only if it looks like a plate

it returns the vehicle type (`unknown` if YOLO missed it), the plate (or `None`), the area it read from, and where the plate text was (for drawing in step 11).

**why the plate zone:** YOLO here is trained on photos of real vehicles. the vehicles in this video are drawings, and YOLO mostly doesn't recognize them (it calls them things like "traffic light" or "tv"). on real camera footage YOLO would do the finding. step 6 prints how often YOLO found the vehicle, so you can see this for yourself.

the reader does **not** fix lookalike characters (O and 0, I and 1, B and 8). if it did, the near-miss plates in the video would open the door.

In [ ]:
OCR_CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-"


def clean(text):
    text = text.upper().replace(" ", "").replace("_", "-")
    if "-" not in text and len(text) in (6, 7):      # OCR sometimes drops the dash
        text = text[:3] + "-" + text[3:]
    return text


def read_plate(frame):
    result = yolo(frame, verbose=False)[0]
    best = None
    for box in result.boxes:
        cls = int(box.cls)
        if cls not in VEHICLES or float(box.conf) < 0.25:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        area = (x2 - x1) * (y2 - y1)
        if best is None or area > best[0]:
            best = (area, VEHICLES[cls], (x1, y1, x2, y2))
    if best is None:
        h, w = frame.shape[:2]
        x1, y1, x2, y2 = (int(w * PLATE_ZONE[0]), int(h * PLATE_ZONE[1]),
                          int(w * PLATE_ZONE[2]), int(h * PLATE_ZONE[3]))
        vehicle = "unknown"
        ox, oy = x1, y1
    else:
        _, vehicle, (x1, y1, x2, y2) = best
        ox, oy = x1, y1 + int((y2 - y1) * 0.55)
    crop = frame[oy:y2, ox:x2]
    plate, plate_box = None, None
    crop = cv2.resize(crop, None, fx=OCR_SCALE, fy=OCR_SCALE, interpolation=cv2.INTER_AREA)
    for pts, text, conf in ocr.readtext(crop, allowlist=OCR_CHARS):
        text = clean(text)
        if PLATE_PATTERN.match(text) and conf > 0.3:
            plate = text
            xs = [p[0] / OCR_SCALE + ox for p in pts]
            ys = [p[1] / OCR_SCALE + oy for p in pts]
            plate_box = (int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys)))
            break
    return vehicle, plate, (x1, y1, x2, y2), plate_box

## 5. try it on one frame

grabs the frame at 2.5 seconds (the first truck, stopped at the door) and shows what YOLO found and what OCR read.
change `AT` to look at other vehicles. `plate_clear_from_s` in ground_truth.csv tells you when each one is stopped.

In [ ]:
from IPython.display import display
from PIL import Image

AT = 2.5

cap = cv2.VideoCapture(VIDEO)
cap.set(cv2.CAP_PROP_POS_MSEC, AT * 1000)
ok, frame = cap.read()
cap.release()

vehicle, plate, box, plate_box = read_plate(frame)
print("vehicle:", vehicle, "  plate:", plate)

shown = frame.copy()
if box:
    cv2.rectangle(shown, box[:2], box[2:], (0, 200, 0), 4)
    cv2.putText(shown, f"{vehicle} {plate}", (box[0], box[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 200, 0), 3)
if plate_box:
    cv2.rectangle(shown, plate_box[:2], plate_box[2:], (0, 220, 255), 4)
display(Image.fromarray(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB)).resize((960, 540)))

## 6. watch the whole video

reads every `SAMPLE_EVERY`th frame and keeps a row for each one. on a laptop CPU this takes a few minutes. OCR is the slow part, not YOLO.

In [ ]:
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

reads = []
started = time.time()
for f in range(0, total, SAMPLE_EVERY):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, frame = cap.read()
    if not ok:
        break
    vehicle, plate, _, plate_box = read_plate(frame)
    reads.append({"frame": f, "t": round(f / fps, 2), "vehicle": vehicle, "plate": plate, "plate_box": plate_box})
    if len(reads) % 60 == 0:
        print(f"{f / fps:5.1f}s of {total / fps:.0f}s")
cap.release()

reads = pd.DataFrame(reads)
print(f"done in {time.time() - started:.0f}s. {len(reads)} frames looked at")
print(f"YOLO found a vehicle on {(reads.vehicle != 'unknown').sum()}, plate read on {reads.plate.notna().sum()}")
reads[reads.plate.notna()].head(20)

## 7. turn reads into arrivals

one good read isn't enough, OCR can be wrong for a single frame. a plate counts as an arrival when it's read the same way `CONFIRM` times in a row (2 looks, about 2/3 of a second apart).
after that, the same plate doesn't count again until a different plate shows up. that stops one truck from opening the door 10 times.

In [ ]:
def find_arrivals(reads, confirm=CONFIRM):
    arrivals = []
    run_plate, run_len, last = None, 0, None
    for r in reads.itertuples():
        if r.plate is None or (isinstance(r.plate, float)):    # no read: keep the streak going, don't add to it
            continue
        if r.plate == run_plate:
            run_len += 1
        else:
            run_plate, run_len = r.plate, 1
        if run_len == confirm and r.plate != last:
            arrivals.append({"t": r.t, "frame": r.frame, "vehicle": r.vehicle, "plate": r.plate})
            last = r.plate
    return pd.DataFrame(arrivals)


arrivals = find_arrivals(reads)
arrivals

## 8. check each arrival against the schedule

these are the same functions as in `dock_shipments.ipynb`, with `source="yolo"`.

in test mode the database is a copy held in memory, so dock.db on disk doesn't change.
in live mode it writes to dock.db and pulses `remote_open`. the door only opens from shut, so in live mode run door_sim.ipynb and close the door between trucks, or most pulses will do nothing.

In [ ]:
disk = sqlite3.connect(DB_PATH)
if LIVE:
    conn = disk
else:
    conn = sqlite3.connect(":memory:")
    disk.backup(conn)
    disk.close()
conn.row_factory = sqlite3.Row

# what the schedule looks like before we start. used for scoring in step 9
start_status = {r["truck_id"]: r["status"] for r in conn.execute("SELECT truck_id, status FROM shipments")}


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def find_truck(truck_id):
    return conn.execute(
        "SELECT * FROM shipments WHERE truck_id = ? AND status IN ('scheduled','late')",
        (truck_id,)).fetchone()


def log_event(event, shipment_id=None, source="yolo", note=None, door=1):
    conn.execute(
        "INSERT INTO door_events (ts, door, event, shipment_id, source, note) VALUES (?,?,?,?,?,?)",
        (now_str(), door, event, shipment_id, source, note))
    conn.commit()


def mark_arrived(shipment_id, source="yolo"):
    conn.execute("UPDATE shipments SET status='arrived', arrived_at=? WHERE shipment_id=?",
                 (now_str(), shipment_id))
    conn.commit()
    log_event("truck_arrived", shipment_id, source)


def open_door():
    if not LIVE:
        return True                       # test mode: pretend it worked
    from pymodbus.client import ModbusTcpClient
    plc = ModbusTcpClient(os.environ.get("PLC_HOST", "127.0.0.1"), port=int(os.environ.get("PLC_PORT", "1502")))
    if not plc.connect():
        print("could not connect to the plc")
        return False
    plc.write_coil(8194, True)            # remote_open %MX0.2
    time.sleep(0.5)
    plc.write_coil(8194, False)
    plc.close()
    return True


def truck_at_door(plate, vehicle, source="yolo"):
    log_event("truck_seen", source=source, note=f"trailer {plate} ({vehicle})")
    s = find_truck(plate)
    if s is None:
        log_event("door_denied", source=source, note=f"{plate} not on schedule")
        return "deny", None
    mark_arrived(s["shipment_id"], source)
    log_event("door_open_request", s["shipment_id"], source)
    if open_door():
        log_event("door_open_sent", s["shipment_id"], source)
    return "open", s["shipment_id"]


decisions, details = [], []
for a in arrivals.itertuples():
    door, shipment = truck_at_door(a.plate, a.vehicle)
    decisions.append(door)
    if shipment:
        s = conn.execute("SELECT carrier, direction, pallets FROM shipments WHERE shipment_id = ?", (shipment,)).fetchone()
        details.append(f"{shipment} · {s['carrier']} · {s['direction']} · {s['pallets']} pallets")
    else:
        details.append(f"{a.plate} not on schedule")
    print(f"{a.t:5.1f}s  {a.vehicle:6s} {a.plate:9s} -> {door}  {details[-1]}")
arrivals["door"] = decisions
arrivals["detail"] = details

## 9. score it

lines each vehicle in the answer key up with the arrival that happened during its time slot.

- **plate ok**: OCR read the right text
- **door ok**: the door did what it should

"should" comes from the schedule as it was when step 8 started, not from the `expected_door` column in the csv.
that column assumes TRL-125 has already arrived. if you reset the day (step 9 in dock_shipments) TRL-125 should open, and this scoring follows along.

In [ ]:
truth = pd.read_csv(GROUND_TRUTH)

rows = []
for t in truth.itertuples():
    hit = arrivals[(arrivals.t >= t.start_s) & (arrivals.t < t.end_s)] if len(arrivals) else arrivals
    got = hit.iloc[0] if len(hit) else None
    should = "open" if start_status.get(t.plate) in ("scheduled", "late") else "deny"
    door = got["door"] if got is not None else "deny"       # nothing read = door stays shut
    rows.append({
        "slot": t.slot, "vehicle": t.vehicle, "plate": t.plate,
        "read": got["plate"] if got is not None else "(none)",
        "yolo_saw": got["vehicle"] if got is not None else "",
        "plate_ok": got is not None and got["plate"] == t.plate,
        "should": should, "door": door, "door_ok": door == should,
        "note": t.note,
    })
score = pd.DataFrame(rows)

print(f"plates read right: {score.plate_ok.sum()} of {len(score)}")
print(f"door right:        {score.door_ok.sum()} of {len(score)}")
extra = len(arrivals) - sum(r["read"] != "(none)" for r in rows)
if extra > 0:
    print(f"extra arrivals that didn't line up with a vehicle: {extra}")
score

## 10. the door log

everything this run wrote, newest last. in test mode it's the in-memory copy.

In [ ]:
pd.read_sql_query("SELECT * FROM door_events WHERE source = 'yolo' ORDER BY event_id", conn)

## 11. make the result video (for recording)

writes `plate_video/dock_cam_result.mp4`: the camera video with what the AI saw and what the door did drawn on top.

- **yellow box**: OCR is reading a plate, not confirmed yet
- **green box**: confirmed, on the schedule, door opens
- **red box**: confirmed, not on the schedule, door stays shut
- **bottom bar**: the door decision with the shipment from dock.db
- **right side**: the door log, newest at the bottom

it only draws what steps 6 to 8 already found, so it's quick (under a minute). run steps 6 to 8 first.

In [ ]:
from PIL import ImageDraw, ImageFont
import numpy as np

OUT = "plate_video/dock_cam_result.mp4"


def font(size):
    try:
        return ImageFont.load_default(size=size)
    except TypeError:                      # older Pillow
        return ImageFont.load_default()


F_BIG, F_MED, F_SMALL = font(44), font(30), font(24)
GREEN, RED, YELLOW, WHITE = (40, 200, 90), (230, 60, 60), (250, 200, 40), (245, 245, 245)

by_frame = {r.frame: r for r in reads.itertuples()}
events = list(arrivals.itertuples())

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

last = None
f = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if f in by_frame:
        last = by_frame[f]
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(img, "RGBA")

    # the pipeline, so viewers know what they're looking at
    d.rectangle([0, 80, w, 128], fill=(0, 0, 0, 150))
    d.text((40, 88), "camera  >  YOLO + OCR (plate)  >  schedule (dock.db)  >  PLC remote_open (door 1)",
           font=F_SMALL, fill=WHITE)

    done = [e for e in events if e.frame <= f]
    current = done[-1] if done else None

    # box on the plate
    if last is not None and last.plate and last.plate_box and f - last.frame < SAMPLE_EVERY * 2:
        confirmed = current is not None and current.plate == last.plate
        color = (GREEN if current.door == "open" else RED) if confirmed else YELLOW
        x1, y1, x2, y2 = last.plate_box
        d.rectangle([x1 - 14, y1 - 14, x2 + 14, y2 + 14], outline=color, width=6)
        label = last.plate if confirmed else f"reading {last.plate}"
        d.rectangle([x1 - 14, y1 - 66, x1 + 22 + d.textlength(label, font=F_MED), y1 - 16], fill=color + (230,))
        d.text((x1 - 4, y1 - 62), label, font=F_MED, fill=(15, 15, 15))

    # door decision bar
    if current is not None and f - current.frame < fps * 4:
        color = GREEN if current.door == "open" else RED
        word = "OPEN DOOR 1" if current.door == "open" else "DOOR STAYS SHUT"
        d.rectangle([0, h - 110, w, h], fill=color + (225,))
        d.text((40, h - 92), f"{current.plate}  >  {word}", font=F_BIG, fill=(15, 15, 15))
        d.text((900, h - 84), current.detail, font=F_MED, fill=(15, 15, 15))

    # door log
    d.rectangle([w - 520, 150, w - 20, 150 + 46 + 40 * 8], fill=(0, 0, 0, 160))
    d.text((w - 500, 160), "door log (source: yolo)", font=F_SMALL, fill=WHITE)
    for i, e in enumerate(done[-7:]):
        d.text((w - 500, 206 + i * 40), f"{e.t:5.1f}s  {e.plate:<9}  {e.door}", font=F_SMALL,
               fill=GREEN if e.door == "open" else RED)

    out.write(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR))
    f += 1

cap.release()
out.release()
print("saved", OUT)

## recording the demo

an order that shows the whole chain, AI side to PLC side:

1. **the camera**: play `plate_video/dock_cam_plates.mp4`. this is what the dock camera sees
2. **the schedule**: in `dock_shipments.ipynb`, step 4 shows today's trucks. point at the `truck_id` column
3. **the AI**: here, step 5 on one frame. YOLO's box, the plate box, the text OCR read
4. **the result**: play `plate_video/dock_cam_result.mp4`. green opens, red stays shut, and the near misses (TRL-I95, TRL-528, TRL-44B) get turned away
5. **the score**: step 9. 16 of 16 door calls right
6. **the PLC**: set `LIVE = True`, run door_sim.ipynb in another tab, open the OpenPLC Editor debugger, and run steps 8 to 10. the first truck on the schedule pulses `remote_open` and the door goes up with `ajar_lamp`, then `open_lamp`

for part 6, start with the door shut. the ladder has no remote close, so only the first open shows on the door. later trucks still get logged.

## things to try

- set `CONFIRM = 1` and see if single bad reads start opening the door
- the near misses (TRL-I95, TRL-528, TRL-44B) are there to see if OCR "fixes" a character into a match. if one of them opens the door, that's the finding
- change `SAMPLE_EVERY` to trade speed for more reads per truck
- swap `yolo11n.pt` for `yolo11s.pt` (bigger, slower, usually finds more)